# Prueba tecnica

**Requisitos antes de correr este notebook**
1. Instalar Ollama: https://ollama.com/download
2. ollama pull llama3.2:3b
3. ollama pull nomic-embed-text
4. pip install ollama pandas numpy



In [2]:
import pandas as pd
import numpy as np
import re
import ollama

pd.set_option('display.max_colwidth', 120)

MODELO_LLM = "llama3.2:3b"
MODELO_EMBEDDINGS = "nomic-embed-text"


## Paso 1 — Cargar y trocear los documentos de referencia

In [4]:
with open("data/documentos_referencia.txt", "r", encoding="utf-8") as f:
    raw = f.read()

parts = re.split(r"\n\[([A-Z_]+)\]\n", raw)
chunks = []
for i in range(1, len(parts), 2):
    chunks.append({"chunk_id": parts[i], "text": parts[i + 1].strip()})

chunks_df = pd.DataFrame(chunks)
chunks_df


,chunk_id,text
0,COMO_SOLICITAR,"Cómo solicitar un producto o servicio, pasos:\n1. Ingresar al portal interno y abrir el formulario F-01 (Solicitud d..."
1,DEVOLUCIONES_GARANTIA,Política de devoluciones y garantías:\nPlazo de devolución: 15 días calendario desde la recepción del producto.\nReq...
2,DESCUENTOS,Política de descuentos por volumen del pedido:\nPedidos de 50 unidades o más: 5% de descuento.\nPedidos de 100 unida...
3,FAQ,"Preguntas frecuentes:\n¿Puedo cancelar una solicitud? Sí, mientras no haya sido aprobada, desde el portal interno.\n..."


## Paso 2 — Índice de embeddings (se calcula la primera vez)

Cada chunk se convierte en un vector con `nomic-embed-text`. Esto se hace una
sola vez al arrancar; no se repite en cada consulta.


In [5]:
def obtener_embedding(texto):
    resp = ollama.embeddings(model=MODELO_EMBEDDINGS, prompt=texto)
    return np.array(resp["embedding"])

chunks_df["embedding"] = chunks_df["text"].apply(obtener_embedding)
print("Índice listo:", len(chunks_df), "chunks vectorizados")


Índice listo: 4 chunks vectorizados


## Paso 3 — Retrieval x cada consuta

Se calcula el embedding de la consulta y se compara contra el índice con
similitud coseno. Devuelve el chunk más parecido y qué tan cerca de 1 está.


In [6]:
def buscar_chunk_mas_similar(consulta):
    q_emb = obtener_embedding(consulta)
    sims = chunks_df["embedding"].apply(
        lambda e: np.dot(q_emb, e) / (np.linalg.norm(q_emb) * np.linalg.norm(e))
    )
    idx = sims.idxmax()
    return chunks_df.loc[idx, "chunk_id"], chunks_df.loc[idx, "text"], sims[idx]

# Prueba
print(buscar_chunk_mas_similar("¿cuánto tiempo tengo para regresar un pedido?"))


('DESCUENTOS', 'Política de descuentos por volumen del pedido:\nPedidos de 50 unidades o más: 5% de descuento.\nPedidos de 100 unidades o más: 10% de descuento.\nDescuentos mayores a estos o condiciones especiales requieren aprobación de la jefatura comercial, no son automáticos.\nLas promociones temporales se publican en el sistema comercial y no se incluyen en este documento.', np.float64(0.5993823440584515))


## Paso 4 — Clasificador usando el LLM directamente

En vez de comparar por similitud contra frases de ejemplo, le pedimos al LLM
que razone la categoría directamente (few-shot en el propio prompt). Consultas
de 1-2 palabras se marcan `AMBIGUA` sin gastar una llamada al modelo.


In [ ]:
CATEGORIAS_VALIDAS = {"RESPONDE_DOCUMENTO", "INFO_TEMPORAL", "REQUIERE_HUMANO", "FUERA_DE_ALCANCE", "AMBIGUA"}

CLASSIFIER_SYSTEM_PROMPT = '''
Clasificas consultas de un asistente comercial de Valtx en EXACTAMENTE una
de estas 5 categorias. Responde UNICAMENTE con el nombre de la categoria,
nada mas.

RESPONDE_DOCUMENTO: la responde el catalogo, como solicitar, devoluciones, garantia o descuentos.
  Ejemplo: "¿cual es el plazo de devolucion?" -> RESPONDE_DOCUMENTO
INFO_TEMPORAL: precio, stock, promocion, tipo de cambio, fechas que cambian.
  Ejemplo: "¿tienen stock del producto alfa?" -> INFO_TEMPORAL
REQUIERE_HUMANO: excepciones, negociaciones, quejas, casos especiales.
  Ejemplo: "necesito un descuento mayor, es urgente" -> REQUIERE_HUMANO
FUERA_DE_ALCANCE: no es tema de negocio (RRHH, TI, cultura general, etc.)
  Ejemplo: "¿como reseteo mi contraseña?" -> FUERA_DE_ALCANCE
AMBIGUA: muy vaga o sin tema claro.
  Ejemplo: "hola tengo una duda" -> AMBIGUA
'''.strip()

def clasificar(consulta):
    resp = ollama.chat(
        model=MODELO_LLM,
        messages=[
            {"role": "system", "content": CLASSIFIER_SYSTEM_PROMPT},
            {"role": "user", "content": consulta},
        ],
        options={"temperature": 0},
    )
    categoria = resp["message"]["content"].strip().upper()
    return categoria if categoria in CATEGORIAS_VALIDAS else "AMBIGUA"

# Pruebas, incluyendo casos nunca vistos
for q in ["aceptan pagos con criptomonedas?", "cuanto es 1-0?", "cual es el plazo de devolucion?", "quiero un descuentazo especial"]:
    print(q, "->", clasificar(q))


aceptan pagos con criptomonedas? -> REQUIERE_HUMANO
cuanto es 1-0? -> AMBIGUA
cual es el plazo de devolucion? -> RESPONDE_DOCUMENTO
quiero un descuentazo especial -> REQUIERE_HUMANO


## Paso 5 — Generación de la respuesta

Solo se llega aquí si la categoría es `RESPONDE_DOCUMENTO` y la similitud del
retrieval pasa el umbral. El LLM responde usando ÚNICAMENTE el chunk recuperado.


In [ ]:
UMBRAL_RETRIEVAL = 0.55  # ajustar segun pruebas con embeddings reales

SYSTEM_PROMPT_PRODUCCION = '''
Eres el asistente de consultas comerciales de Valtx. Respondes UNICAMENTE
usando el fragmento de contexto proporcionado. No uses conocimiento general
aunque lo tengas. Si el fragmento no cubre la pregunta, dilo explicitamente.
Responde en espanol, breve y claro (maximo 3-4 oraciones).
'''.strip()

def generar_respuesta(consulta, chunk_text):
    resp = ollama.chat(
        model=MODELO_LLM,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_PRODUCCION},
            {"role": "user", "content": f"Contexto:\n{chunk_text}\n\nPregunta: {consulta}"},
        ],
        options={"temperature": 0.2},
    )
    return resp["message"]["content"]


## Paso 6 — Groundedness check (segunda verificación)

Antes de entregar la respuesta, le preguntamos al LLM si TODO lo que dijo
está respaldado por el contexto. Si dice que no, se descarta y se escala.


In [21]:
import unicodedata

def quitar_tildes(texto):
    return ''.join(c for c in unicodedata.normalize('NFKD', texto) if not unicodedata.combining(c))

UMBRAL_GROUNDEDNESS = 0.60 

def esta_anclada(respuesta, chunk_text):
    emb_respuesta = obtener_embedding(respuesta)
    emb_chunk = obtener_embedding(chunk_text)
    similitud = np.dot(emb_respuesta, emb_chunk) / (np.linalg.norm(emb_respuesta) * np.linalg.norm(emb_chunk))
    return similitud >= UMBRAL_GROUNDEDNESS, similitud


## Paso 7 — Pipeline completo

In [22]:
RESPUESTAS_FIJAS = {
    "INFO_TEMPORAL": "Ese dato (precio, stock o promoción) se gestiona en el sistema comercial y cambia con frecuencia. Te derivo con el equipo comercial para confirmar el valor vigente.",
    "REQUIERE_HUMANO": "Este caso requiere evaluación de una persona del equipo comercial. Te voy a derivar con el equipo.",
    "FUERA_DE_ALCANCE": "Esa consulta no corresponde a productos, procesos ni políticas comerciales. Si es un tema interno, contacta al área correspondiente.",
    "AMBIGUA": "No tengo claro a qué te refieres. ¿Podrías dar más detalle (catálogo, solicitud, devolución, garantía o descuentos)?",
}

def pipeline(consulta):
    categoria = clasificar(consulta)

    if categoria != "RESPONDE_DOCUMENTO":
        return {"consulta": consulta, "categoria": categoria, "motivo": "router", "respuesta": RESPUESTAS_FIJAS[categoria]}

    chunk_id, chunk_text, score = buscar_chunk_mas_similar(consulta)

    if score < UMBRAL_RETRIEVAL:
        return {"consulta": consulta, "categoria": "REQUIERE_HUMANO", "motivo": f"similitud baja ({score:.2f})", "respuesta": RESPUESTAS_FIJAS["REQUIERE_HUMANO"]}

    respuesta = generar_respuesta(consulta, chunk_text)

    anclada, sim_groundedness = esta_anclada(respuesta, chunk_text)
    if not anclada:
        return {"consulta": consulta, "categoria": "REQUIERE_HUMANO", "motivo": f"no anclada al contexto (sim={sim_groundedness:.2f})", "respuesta": RESPUESTAS_FIJAS["REQUIERE_HUMANO"]}

    return {"consulta": consulta, "categoria": "RESPONDE_DOCUMENTO", "motivo": f"chunk={chunk_id}, score={score:.2f}, groundedness={sim_groundedness:.2f}", "respuesta": respuesta}

# Prueba con un par de casos
for q in ["¿Cuál es el plazo para devolver un producto?", "¿aceptan criptomonedas?"]:
    r = pipeline(q)
    print(r["categoria"], "|", r["motivo"], "|", r["respuesta"][:100])
    print("---")


RESPONDE_DOCUMENTO | chunk=DEVOLUCIONES_GARANTIA, score=0.78, groundedness=0.84 | El plazo para devolver un producto es de 15 días calendario desde la recepción del producto.
---
REQUIERE_HUMANO | router | Este caso requiere evaluación de una persona del equipo comercial. Te voy a derivar con el equipo.
---


## Paso 8 — Correr sobre las 80 consultas reales del CSV

In [23]:
consultas_df = pd.read_csv("data/consultas.csv", sep=";", encoding="utf-8-sig")

resultados = consultas_df["consulta"].apply(pipeline).apply(pd.Series)
final_df = pd.concat([consultas_df[["id", "fecha", "canal", "consulta"]], resultados.drop(columns=["consulta"])], axis=1)
final_df.to_csv("data/consultas_clasificadas_final.csv", index=False)
final_df.head(15)


,id,fecha,canal,consulta,categoria,motivo,respuesta
0,C001,11/02/2026,correo,¿Cómo solicito un producto del catálogo?,RESPONDE_DOCUMENTO,"chunk=COMO_SOLICITAR, score=0.80, groundedness=0.89","Para solicitar un producto del catálogo, debes ingresar al portal interno de Valtx, abrir el formulario F-01 (Solici..."
1,C002,13/02/2026,chat,¿Cuáles son los pasos para pedir un servicio nuevo?,RESPONDE_DOCUMENTO,"chunk=COMO_SOLICITAR, score=0.66, groundedness=0.93","Para pedir un servicio nuevo, sigue estos pasos:\n\n1. Ingresar al portal interno y abrir el formulario F-01 (Solici..."
2,C003,18/02/2026,formulario,¿Dónde encuentro el formulario de solicitud?,RESPONDE_DOCUMENTO,"chunk=COMO_SOLICITAR, score=0.73, groundedness=0.82","Puedes encontrar el formulario F-01 de solicitud de producto/servicio en el portal interno de Valtx, específicamente..."
3,C004,20/02/2026,correo,¿Qué información debo incluir en una solicitud?,RESPONDE_DOCUMENTO,"chunk=FAQ, score=0.73, groundedness=0.68",No se proporciona información específica sobre qué información debe incluir una solicitud en el fragmento de context...
4,C005,25/02/2026,chat,¿Puedo cancelar una solicitud ya enviada?,RESPONDE_DOCUMENTO,"chunk=FAQ, score=0.82, groundedness=0.84","Sí, puedes cancelar una solicitud ya enviada, siempre y cuando no haya sido aprobada."
5,C006,2/03/2026,correo,¿Cuál es el plazo para devolver un producto?,RESPONDE_DOCUMENTO,"chunk=DEVOLUCIONES_GARANTIA, score=0.78, groundedness=0.84",El plazo para devolver un producto es de 15 días calendario desde la recepción del producto.
6,C007,4/03/2026,teléfono,¿Qué necesito para hacer una devolución?,RESPONDE_DOCUMENTO,"chunk=DEVOLUCIONES_GARANTIA, score=0.73, groundedness=0.75","Para realizar una devolución, necesitas: 1) el producto sin uso, 2) con empaque original y 3) el número de solicitud..."
7,C008,6/03/2026,chat,¿Cómo funciona la garantía de los productos?,RESPONDE_DOCUMENTO,"chunk=DEVOLUCIONES_GARANTIA, score=0.72, groundedness=0.88",La garantía de los productos de Valtx cubre defectos de fábrica durante 12 meses desde la fecha de compra. Para recl...
8,C009,9/03/2026,correo,¿Qué descuento aplica para pedidos de 60 unidades?,RESPONDE_DOCUMENTO,"chunk=DESCUENTOS, score=0.74, groundedness=0.89","Según la política de descuentos por volumen, un pedido de 60 unidades no alcanza el umbral de 50 unidades, por lo qu..."
9,C010,11/03/2026,chat,¿Y para un pedido de 120 unidades cuánto de descuento hay?,RESPONDE_DOCUMENTO,"chunk=DESCUENTOS, score=0.77, groundedness=0.80","Para un pedido de 120 unidades, se aplica un descuento del 10%."


## Paso 9 — Resumen ejecutivo

In [ ]:
resumen = final_df["categoria"].value_counts()
resumen_pct = (resumen / len(final_df) * 100).round(1)
print("Distribución de categorías sobre las 80 consultas:")
for cat in resumen.index:
    print(f"  {cat:20s} {resumen[cat]:3d} consultas  ({resumen_pct[cat]}%)")
